# 29. 확신도 기반 mean_working 폴백 SVR

26번은 `mean_working` 을 완전히 제외했다. 거리 피처로 넣으면 CV 가 0.1686 까지 나빠졌기 때문이다.
그런데 값별 타겟 평균을 보면 0.28(=6) 에서 0.78(=12) 까지 퍼져 있다. 신호가 없는 게 아니라
**커널 거리 안에 넣을 자리가 아니었던 것**이다.

여기서는 `mean_working` 을 거리 계산 밖으로 빼서 별도 경로로 쓴다.

1. SVR 예측은 26번 그대로 둔다
2. `mean_working` 레벨별 조건부 평균을 따로 만든다 (표본 적은 레벨은 경험적 베이즈로 축소)
3. **SVR 예측이 중앙값 근처면** 모델이 정보가 없다는 뜻이므로 레벨 추정치로 되돌린다

3번의 확신도는 모델 자기 출력(`|예측 − 중앙값|`)만으로 계산한다.

## 1. 설정

In [1]:
import warnings
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer, RobustScaler
from sklearn.svm import SVR

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
SEEDS = [42, 2024, 7]

NUM = ['age', 'height', 'weight', 'cholesterol', 'systolic_blood_pressure',
       'diastolic_blood_pressure', 'glucose', 'bone_density']
CAT = ['gender', 'activity', 'smoke_status', 'medical_history',
       'family_medical_history', 'sleep_pattern', 'edu_level']

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
y = train['stress_score'].to_numpy(dtype=float)
print(train.shape, test.shape)

(3000, 18) (3000, 17)


## 2. mean_working 레벨 신호

26번이 버린 변수를 다시 본다. 값별 타겟 평균이다.

In [2]:
lv = train['mean_working'].round(0)
tbl = train.groupby(lv)['stress_score'].agg(['count', 'mean']).round(4)
print(tbl.to_string())
print(f"\n전체 타겟 평균 {y.mean():.4f}, 중앙값 {np.median(y):.4f}")
print(f"결측 {train['mean_working'].isna().sum()}행")

              count    mean
mean_working               
4.0               5  0.2780
5.0              20  0.3055
6.0              94  0.3148
7.0             318  0.4657
8.0             451  0.4823
9.0             537  0.4611
10.0            346  0.4677
11.0            120  0.5964
12.0             26  0.7746
13.0             23  0.7357
14.0              9  0.7222
15.0             17  0.6376
16.0              2  0.6600

전체 타겟 평균 0.4821, 중앙값 0.4800
결측 1032행


4~6 시간대가 0.28~0.32, 11~16 시간대가 0.60~0.78 이다. 전체 평균 0.482 대비 위아래로 크게 벌어진다.
26번이 이 변수를 버린 판단은 **거리 피처로서는** 옳았지만, 신호 자체는 있다.

## 3. 전처리와 모델

26번과 동일하다. `mean_working` 은 여전히 거리 계산에 넣지 않는다.
스케일러와 인코더는 train 으로만 fit 하고 test 에는 transform 만 적용한다.

In [3]:
def numeric(df):
    x = df[NUM].copy()
    x['bmi'] = (df['weight'] / (df['height'] / 100) ** 2).round(2)
    return x.to_numpy(dtype=float)

scaler = RobustScaler().fit(numeric(train))
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False,
                    dtype=float).fit(train[CAT].fillna('Unknown'))

def build(df):
    return np.hstack([scaler.transform(numeric(df)),
                      ohe.transform(df[CAT].fillna('Unknown'))])

X, X_test = build(train), build(test)
LVL = train['mean_working'].round(0).fillna(-1).to_numpy(float)
LVL_TEST = test['mean_working'].round(0).fillna(-1).to_numpy(float)
MEDIAN = float(np.median(y))

def model():
    return TransformedTargetRegressor(
        regressor=SVR(C=4.0, gamma=2.0, kernel='rbf', epsilon=0.0),
        transformer=QuantileTransformer(output_distribution='normal',
                                        n_quantiles=1000, random_state=RANDOM_STATE))

print(f'X {X.shape}, X_test {X_test.shape}, 타겟 중앙값 {MEDIAN}')

X (3000, 32), X_test (3000, 32), 타겟 중앙값 0.48


## 4. mean_working 레벨표 (경험적 베이즈 축소)

레벨별 단순 평균을 그대로 믿으면 표본이 적은 레벨에서 과적합한다.
`EB(v) = n_v/(n_v+k) * mean(y|v) + k/(n_v+k) * mean(y)` 로 전체 평균 쪽으로 당긴다.
`k` 가 클수록 보수적이다. 레벨표는 **학습 폴드 안에서만** 만든다.

In [4]:
EB_K = 20

def eb_table(lv_tr, y_tr, lv_target, k=EB_K):
    g = pd.DataFrame({'l': lv_tr, 'y': y_tr}).groupby('l')['y'].agg(['count', 'mean'])
    gm = y_tr.mean()
    eb = (g['count'] * g['mean'] + k * gm) / (g['count'] + k)
    return pd.Series(lv_target).map(eb).fillna(gm).to_numpy(float)

demo = eb_table(LVL, y, np.array([4., 6., 9., 11., 12., 16., -1.]))
print('EB 축소 후 레벨 추정치')
for v, e in zip([4, 6, 9, 11, 12, 16, '결측'], demo):
    print(f'   mean_working={str(v):>6}  ->  {e:.4f}')

EB 축소 후 레벨 추정치
   mean_working=     4  ->  0.4413
   mean_working=     6  ->  0.3441
   mean_working=     9  ->  0.4619
   mean_working=    11  ->  0.5801
   mean_working=    12  ->  0.6474
   mean_working=    16  ->  0.4983
   mean_working=    결측  ->  0.4910


## 5. 확신도 기반 폴백

SVR 예측이 중앙값에서 멀면 모델이 근거를 찾은 것이고, 중앙값 근처면 정보가 없다는 뜻이다.
후자에만 레벨 추정치를 섞는다.

`w = exp(-(|예측 - 중앙값| / tau)^2)`

`tau` 가 작을수록 "중앙값 바로 근처"로 좁게 본다. 모델 자기 출력만 쓰므로 추가 정보가 필요없다.

In [5]:
TAU = 0.02

def blend(pred_svr, pred_level, tau=TAU):
    w = np.exp(-((np.abs(pred_svr - MEDIAN) / tau) ** 2))
    return np.clip((1 - w) * pred_svr + w * pred_level, 0, 1)

def oof(seed):
    ps, pl = np.zeros(len(y)), np.zeros(len(y))
    for t, v in KFold(5, shuffle=True, random_state=seed).split(X):
        ps[v] = np.clip(model().fit(X[t], y[t]).predict(X[v]), 0, 1)
        pl[v] = eb_table(LVL[t], y[t], LVL[v])
    return ps, pl

ps42, pl42 = oof(42)
w42 = np.exp(-((np.abs(ps42 - MEDIAN) / TAU) ** 2))
print(f'가중치 w 분포 : w>0.5 인 행 {(w42 > 0.5).sum()}개 ({(w42 > 0.5).mean()*100:.1f}%)')
print(f'                w<0.1 인 행 {(w42 < 0.1).sum()}개 ({(w42 < 0.1).mean()*100:.1f}%)')
print(f'예측 표준편차 {ps42.std():.4f} vs 타겟 {y.std():.4f}')

가중치 w 분포 : w>0.5 인 행 1740개 (58.0%)
                w<0.1 인 행 1177개 (39.2%)
예측 표준편차 0.1852 vs 타겟 0.2882


확신도가 양극화돼 있다. 대부분의 행은 w 가 0 이거나 1 에 가깝고 중간이 적다.
모델이 "아는 행"과 "모르는 행"이 뚜렷하게 갈린다는 뜻이다.

## 6. 검증

시드 3개로 대응비교한다. 같은 시드, 같은 폴드에서 26번과 비교한다.

In [6]:
print(f'{"시드":<8}{"26번":<12}{"29번":<12}차이')
deltas = []
for s in SEEDS:
    p, l = oof(s)
    a = mean_absolute_error(y, p)
    b = mean_absolute_error(y, blend(p, l))
    deltas.append(b - a)
    print(f'{s:<8}{a:<12.6f}{b:<12.6f}{b - a:+.6f}')
print(f'\n평균 {np.mean(deltas):+.6f}, 부호 일관 {all(d < 0 for d in deltas)}')
print(f'상수 0.50 기준선 {mean_absolute_error(y, np.full_like(y, 0.5)):.6f}')
print(f'상수 {MEDIAN} 기준선 {mean_absolute_error(y, np.full_like(y, MEDIAN)):.6f}')

시드      26번         29번         차이


42      0.147668    0.145707    -0.001961


2024    0.144505    0.142118    -0.002388


7       0.146518    0.144733    -0.001785

평균 -0.002045, 부호 일관 True
상수 0.50 기준선 0.249883
상수 0.48 기준선 0.249443


## 7. tau 민감도

`tau` 를 바꿔가며 안정적인 구간인지 본다.

In [7]:
p, l = oof(42)
base = mean_absolute_error(y, p)
print(f'{"tau":>8}{"MAE":>12}{"차이":>12}')
for t in [0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.12]:
    m = mean_absolute_error(y, blend(p, l, t))
    print(f'{t:>8.3f}{m:>12.6f}{m - base:>+12.6f}{"  *" if m < base else ""}')

     tau         MAE          차이
   0.005    0.146696   -0.000973  *
   0.010    0.146121   -0.001548  *
   0.020    0.145707   -0.001961  *
   0.030    0.145796   -0.001872  *
   0.050    0.146388   -0.001281  *
   0.080    0.147887   +0.000218
   0.120    0.151017   +0.003349


## 8. 최종 학습 및 제출

train 3000행 전체로 SVR 을 학습하고 레벨표도 전체로 만든다.

In [8]:
final_svr = model().fit(X, y)
pred_svr = np.clip(final_svr.predict(X_test), 0, 1)
pred_level = eb_table(LVL, y, LVL_TEST)
pred = blend(pred_svr, pred_level)

w_test = np.exp(-((np.abs(pred_svr - MEDIAN) / TAU) ** 2))
print(f'레벨 추정으로 크게 이동한 행 (w>0.5) : {(w_test > 0.5).sum()}개')
print(f'SVR 예측 유지 (w<0.1)              : {(w_test < 0.1).sum()}개')
print(f'평균 이동폭                        : {np.abs(pred - pred_svr).mean():.4f}')
print(f'예측 평균 {pred.mean():.4f}, 표준편차 {pred.std():.4f}, '
      f'범위 {pred.min():.3f}~{pred.max():.3f}')

sub = pd.read_csv('../data/sample_submission.csv')
sub['stress_score'] = pred
sub.to_csv('../submissions/submit_29_conf_fallback.csv', index=False)
print('\nsaved -> submissions/submit_29_conf_fallback.csv')
print(sub.head().to_string(index=False))

레벨 추정으로 크게 이동한 행 (w>0.5) : 1511개
SVR 예측 유지 (w<0.1)              : 1394개
평균 이동폭                        : 0.0100
예측 평균 0.4947, 표준편차 0.2011, 범위 0.000~1.000

saved -> submissions/submit_29_conf_fallback.csv
       ID  stress_score
TEST_0000      0.490809
TEST_0001      0.970000
TEST_0002      0.190000
TEST_0003      0.473280
TEST_0004      0.529878


## 9. 정리

| 항목 | 26번 | 29번 |
|---|---|---|
| 기본 모델 | SVR C=4.0 gamma=2.0 | 동일 (변경 없음) |
| mean_working | 완전 제외 | 거리 계산에서는 제외, 레벨 폴백으로만 사용 |
| 폴백 조건 | 없음 | `\|예측 − 중앙값\|` 이 작을 때 (모델 자기 출력) |
| 레벨 추정 | — | 경험적 베이즈 축소 (k=20), 학습 폴드 내부에서만 적합 |

26번이 `mean_working` 을 버린 건 **거리 피처로서는** 옳은 판단이었다.
커널 거리에 넣으면 모든 행의 기하가 바뀌어 손해가 신호보다 크다.
거리 밖으로 빼서 확신 없는 행에만 쓰면 그 손해가 생기지 않는다.

### 규정 관련

- 스케일러, 인코더, 레벨표 전부 train (또는 학습 폴드) 에서만 적합하고 test 에는 적용만 했다
- 근접 중복행 탐색, 최근접이웃 매칭, test 행 간 정보 공유 코드가 없다
- 폴백 판단은 모델 자기 예측값만 쓴다. 외부 정보가 들어가지 않는다
- 사용한 관계(근로시간 - 스트레스)는 데이터 첫 탐색 때부터 보이던 평범한 상관관계다

다만 26번과 성격이 다른 처리이므로 제출 전 TA 확인을 권장한다.